In [26]:
import inspect
from src.data_loader import load_events

print(inspect.signature(load_events))

(detector='czt1')


In [25]:
import importlib
import src.data_loader

importlib.reload(src.data_loader)

from src.data_loader import *

In [21]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from scipy.signal import find_peaks
from scipy.stats import skew, kurtosis

from src.data_loader import *
from src.preprocessing import *
from src.validation import *

warnings.filterwarnings("ignore")

In [27]:
lightcurve_df = preprocess_lightcurve(
    load_lightcurve("czt1")
)

metadata_df, counts_matrix, stat_err_matrix, channels = load_spectra("czt1")

metadata_df, counts_matrix = preprocess_spectra(
    metadata_df,
    counts_matrix
)

hk_df = preprocess_housekeeping(
    load_housekeeping()
)

events_df = preprocess_events(
    load_events("czt1")
)

print("Lightcurve :", lightcurve_df.shape)
print("Spectra    :", counts_matrix.shape)
print("HK         :", hk_df.shape)
print("Events     :", events_df.shape)


Loaded hk.fits
Loaded CZT1 events
Lightcurve : (43188, 5)
Spectra    : (2158, 341)
HK         : (5610, 62)
Events     : (1303693, 9)


In [28]:
features = lightcurve_df.copy()

print(features.shape)
features.head()

(43188, 5)


,MJD,ISOT,COUNTS,STAT_ERR,DATETIME
0,61215.500020,2026-06-24T12:00:01.739,252.0,15.874508,2026-06-24 12:00:01.739
1,61215.500032,2026-06-24T12:00:02.739,0.0,0.000000,2026-06-24 12:00:02.739
2,61215.500043,2026-06-24T12:00:03.739,0.0,0.000000,2026-06-24 12:00:03.739
3,61215.500055,2026-06-24T12:00:04.739,0.0,0.000000,2026-06-24 12:00:04.739
4,61215.500066,2026-06-24T12:00:05.739,0.0,0.000000,2026-06-24 12:00:05.739


In [29]:
WINDOW = 30

features["rolling_mean"] = (
    features["COUNTS"]
    .rolling(WINDOW, min_periods=1)
    .mean()
)

features["rolling_std"] = (
    features["COUNTS"]
    .rolling(WINDOW, min_periods=1)
    .std()
)

features["rolling_max"] = (
    features["COUNTS"]
    .rolling(WINDOW, min_periods=1)
    .max()
)

features["rolling_min"] = (
    features["COUNTS"]
    .rolling(WINDOW, min_periods=1)
    .min()
)

In [30]:
features["ema_10"] = (
    features["COUNTS"]
    .ewm(span=10)
    .mean()
)

features["ema_30"] = (
    features["COUNTS"]
    .ewm(span=30)
    .mean()
)

In [31]:
features["diff1"] = features["COUNTS"].diff()

features["diff2"] = features["diff1"].diff()

features["gradient"] = np.gradient(
    features["COUNTS"]
)

In [32]:
for lag in [1, 2, 5, 10]:

    features[f"lag_{lag}"] = (
        features["COUNTS"]
        .shift(lag)
    )

In [33]:
features["window_energy"] = (
    features["COUNTS"] ** 2
).rolling(
    WINDOW,
    min_periods=1
).sum()

In [34]:
features["window_variance"] = (
    features["COUNTS"]
    .rolling(WINDOW, min_periods=1)
    .var()
)

In [35]:
features = features.bfill().ffill()

In [36]:
print(features.shape)

print("\nMissing Values")
print(features.isna().sum())

features.head()

(43188, 20)

Missing Values
MJD                0
ISOT               0
COUNTS             0
STAT_ERR           0
DATETIME           0
rolling_mean       0
rolling_std        0
rolling_max        0
rolling_min        0
ema_10             0
ema_30             0
diff1              0
diff2              0
gradient           0
lag_1              0
lag_2              0
lag_5              0
lag_10             0
window_energy      0
window_variance    0
dtype: int64


,MJD,ISOT,COUNTS,STAT_ERR,DATETIME,rolling_mean,rolling_std,rolling_max,rolling_min,ema_10,ema_30,diff1,diff2,gradient,lag_1,lag_2,lag_5,lag_10,window_energy,window_variance
0,61215.500020,2026-06-24T12:00:01.739,252.0,15.874508,2026-06-24 12:00:01.739,252.0,178.190909,252.0,252.0,252.000000,252.000000,-252.0,252.0,-252.0,252.0,252.0,252.0,252.0,63504.0,31752.0
1,61215.500032,2026-06-24T12:00:02.739,0.0,0.000000,2026-06-24 12:00:02.739,126.0,178.190909,252.0,0.0,113.400000,121.800000,-252.0,252.0,-126.0,252.0,252.0,252.0,252.0,63504.0,31752.0
2,61215.500043,2026-06-24T12:00:03.739,0.0,0.000000,2026-06-24 12:00:03.739,84.0,145.492268,252.0,0.0,67.813953,78.464272,0.0,252.0,0.0,0.0,252.0,252.0,252.0,63504.0,21168.0
3,61215.500055,2026-06-24T12:00:04.739,0.0,0.000000,2026-06-24 12:00:04.739,63.0,126.000000,252.0,0.0,45.472277,56.844506,0.0,0.0,0.0,0.0,0.0,252.0,252.0,63504.0,15876.0
4,61215.500066,2026-06-24T12:00:05.739,0.0,0.000000,2026-06-24 12:00:05.739,50.4,112.697826,252.0,0.0,32.418423,43.911005,0.0,0.0,0.0,0.0,0.0,252.0,252.0,63504.0,12700.8


In [38]:
from scipy.stats import skew

WINDOW = 30

features["rolling_skew"] = (
    features["COUNTS"]
    .rolling(WINDOW, min_periods=5)
    .apply(lambda x: skew(x), raw=False)
)

In [39]:
from scipy.stats import kurtosis

features["rolling_kurtosis"] = (
    features["COUNTS"]
    .rolling(WINDOW, min_periods=5)
    .apply(lambda x: kurtosis(x), raw=False)
)

In [40]:
features["rolling_rms"] = (
    features["COUNTS"]**2
).rolling(
    WINDOW,
    min_periods=1
).mean() ** 0.5

In [41]:
features["rolling_median"] = (
    features["COUNTS"]
    .rolling(WINDOW, min_periods=1)
    .median()
)

In [42]:
def rolling_mad(x):
    med = np.median(x)
    return np.median(np.abs(x - med))

features["rolling_mad"] = (
    features["COUNTS"]
    .rolling(WINDOW, min_periods=5)
    .apply(rolling_mad, raw=True)
)

In [43]:
features["rolling_cv"] = (
    features["rolling_std"]
    /
    (features["rolling_mean"] + 1e-8)
)

In [44]:
features["z_score"] = (
    features["COUNTS"]
    -
    features["rolling_mean"]
) / (
    features["rolling_std"] + 1e-8
)

In [45]:
features["robust_z"] = (
    features["COUNTS"]
    -
    features["rolling_median"]
) / (
    features["rolling_mad"] + 1e-8
)

In [46]:
features["peak_to_peak"] = (
    features["rolling_max"]
    -
    features["rolling_min"]
)

In [47]:
features["local_energy"] = (
    features["COUNTS"]**2
).rolling(
    WINDOW,
    min_periods=1
).sum()

In [48]:
features = features.bfill().ffill()

print(features.isna().sum().sum())

0


In [49]:
stat_cols = [
    "rolling_skew",
    "rolling_kurtosis",
    "rolling_rms",
    "rolling_median",
    "rolling_mad",
    "rolling_cv",
    "z_score",
    "robust_z",
    "peak_to_peak",
    "local_energy"
]

features[stat_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
rolling_skew,43188.0,2.509770e+00,4.026617e-01,1.216142,2.249752,2.452690,2.747907,4.702266e+00
rolling_kurtosis,43188.0,5.185192e+00,2.473954e+00,-0.206379,3.473058,4.670363,6.402435,2.137280e+01
rolling_rms,43188.0,3.592206e+01,7.463851e+00,14.457985,30.747358,35.328223,40.503086,2.520000e+02
rolling_median,43188.0,8.752431e-03,1.355720e+00,0.000000,0.000000,0.000000,0.000000,2.520000e+02
rolling_mad,43188.0,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000e+00
rolling_cv,43188.0,2.607960e+00,3.030392e-01,0.707107,2.404759,2.628102,2.752233,4.045547e+00
z_score,43188.0,-7.449397e-03,9.573988e-01,-0.707107,-0.403971,-0.373527,-0.324381,4.793939e+00
robust_z,43188.0,1.318160e+09,3.415548e+09,0.000000,0.000000,0.000000,0.000000,3.620000e+10
peak_to_peak,43188.0,1.305642e+02,3.286159e+01,0.000000,107.000000,127.000000,150.000000,3.620000e+02
local_energy,43188.0,4.024885e+04,1.669835e+04,6271.000000,28362.000000,37442.500000,49215.000000,1.802650e+05


In [50]:
spectral_features = metadata_df.copy()

spectral_features.head()

,SPEC_NUM,ROWID,TSTART,TSTOP,EXPOSURE,MID_TIME
0,0,Spectrum0,0.0,20.0,20.0,10.0
1,1,Spectrum1,20.0,40.0,20.0,30.0
2,2,Spectrum2,40.0,60.0,20.0,50.0
3,3,Spectrum3,60.0,80.0,20.0,70.0
4,4,Spectrum4,80.0,100.0,20.0,90.0


In [51]:
spectral_features["spec_total_counts"] = counts_matrix.sum(axis=1)

In [52]:
spectral_features["spec_mean_counts"] = counts_matrix.mean(axis=1)

spectral_features["spec_std_counts"] = counts_matrix.std(axis=1)

In [53]:
spectral_features["spec_max"] = counts_matrix.max(axis=1)

spectral_features["spec_min"] = counts_matrix.min(axis=1)

In [54]:
spectral_features["peak_channel"] = np.argmax(
    counts_matrix,
    axis=1
)

In [55]:
channels = np.arange(counts_matrix.shape[1])

spectral_features["spectral_centroid"] = (
    counts_matrix * channels
).sum(axis=1) / (
    counts_matrix.sum(axis=1) + 1e-8
)

In [56]:
centroid = spectral_features["spectral_centroid"].values

spectral_features["spectral_spread"] = np.sqrt(
    (
        counts_matrix
        *
        (channels - centroid[:, None]) ** 2
    ).sum(axis=1)
    /
    (
        counts_matrix.sum(axis=1)
        + 1e-8
    )
)

In [57]:
prob = counts_matrix / (
    counts_matrix.sum(axis=1, keepdims=True)
    + 1e-8
)

spectral_features["spectral_entropy"] = (
    -(prob * np.log2(prob + 1e-12))
    .sum(axis=1)
)

In [58]:
LOW = slice(0,114)

MID = slice(114,227)

HIGH = slice(227,341)

spectral_features["low_energy"] = (
    counts_matrix[:,LOW].sum(axis=1)
)

spectral_features["mid_energy"] = (
    counts_matrix[:,MID].sum(axis=1)
)

spectral_features["high_energy"] = (
    counts_matrix[:,HIGH].sum(axis=1)
)

In [59]:
spectral_features["hardness_ratio1"] = (
    spectral_features["mid_energy"]
    /
    (
        spectral_features["low_energy"]
        + 1e-8
    )
)

spectral_features["hardness_ratio2"] = (
    spectral_features["high_energy"]
    /
    (
        spectral_features["mid_energy"]
        + 1e-8
    )
)

In [60]:
spectral_features["dominant_fraction"] = (
    counts_matrix.max(axis=1)
    /
    (
        counts_matrix.sum(axis=1)
        + 1e-8
    )
)

In [61]:
spectral_features["active_channels"] = (
    counts_matrix > 0
).sum(axis=1)

In [62]:
print(spectral_features.shape)

spectral_features.describe().T

(2158, 22)


,count,mean,std,min,25%,50%,75%,max
SPEC_NUM,2158.0,1078.500000,6.231053e+02,0.000000,539.250000,1078.500000,1617.750000,2157.000000
TSTART,2158.0,21570.000000,1.246211e+04,0.000000,10785.000000,21570.000000,32355.000000,43140.000000
TSTOP,2158.0,21590.000000,1.246211e+04,20.000000,10805.000000,21590.000000,32375.000000,43159.999999
EXPOSURE,2158.0,20.000000,2.142355e-07,19.999999,20.000000,20.000000,20.000000,20.000001
MID_TIME,2158.0,21580.000000,1.246211e+04,10.000000,10795.000000,21580.000000,32365.000000,43149.999999
spec_total_counts,2158.0,507.906860,1.249478e+02,247.000000,401.000000,509.000000,597.000000,1180.000000
spec_mean_counts,2158.0,1.489463,3.664158e-01,0.724340,1.175953,1.492669,1.750733,3.460411
spec_std_counts,2158.0,4.080471,1.239193e+00,1.520233,3.173844,3.951625,4.867901,11.310431
spec_max,2158.0,31.096849,9.900074e+00,11.000000,24.000000,30.000000,37.000000,84.000000
spec_min,2158.0,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000


In [63]:
hk_features = hk_df.copy()

print(hk_features.shape)

hk_features.head()

(5610, 62)


,l0recnum,l0grtyr,l0grtmon,l0grtdy,l0grthr,l0grtmin,l0grtsc,l0grtmsc,l0utcyr,l0utcmon,...,czt2bunpxctr,sunradeg,sundecdeg,suninfov,sun2yawdeg,sun2rolldeg,sun2pitchdeg,yawradeg,yawdecdeg,lastevtmjd
0,1.0,2026.0,6.0,25.0,6.0,26.0,13.0,375.0,2026.0,6.0,...,-9.223372e+18,92.759348,23.447782,1.0,0.176917,89.943452,89.832363,92.940753,23.387868,61215.500014
1,2.0,2026.0,6.0,25.0,6.0,26.0,13.0,669.0,2026.0,6.0,...,-9.223372e+18,92.759351,23.447782,1.0,0.176912,89.943445,89.832371,92.940748,23.387861,61215.500018
2,3.0,2026.0,6.0,25.0,6.0,26.0,13.0,940.0,2026.0,6.0,...,-9.223372e+18,92.759442,23.447780,1.0,0.176972,89.943102,89.832424,92.940774,23.387518,61215.500106
3,4.0,2026.0,6.0,25.0,6.0,26.0,14.0,209.0,2026.0,6.0,...,-9.223372e+18,92.759544,23.447779,1.0,0.176566,89.942890,89.832925,92.940325,23.387312,61215.500220
4,5.0,2026.0,6.0,25.0,6.0,26.0,14.0,456.0,2026.0,6.0,...,-9.223372e+18,92.759636,23.447777,1.0,0.175766,89.943051,89.833716,92.939558,23.387487,61215.500299


In [64]:
hk_features["czt_temp_mean"] = (
    hk_features["czt1temp"] +
    hk_features["czt2temp"]
) / 2

hk_features["cdte_temp_mean"] = (
    hk_features["cdte1temp"] +
    hk_features["cdte2temp"]
) / 2

hk_features["czt_temp_diff"] = (
    hk_features["czt1temp"] -
    hk_features["czt2temp"]
)

hk_features["cdte_temp_diff"] = (
    hk_features["cdte1temp"] -
    hk_features["cdte2temp"]
)

In [65]:
WINDOW = 20

for col in [
    "czt1temp",
    "czt2temp",
    "cdte1temp",
    "cdte2temp"
]:

    hk_features[f"{col}_rolling_std"] = (
        hk_features[col]
        .rolling(WINDOW, min_periods=1)
        .std()
    )

    hk_features[f"{col}_gradient"] = np.gradient(
        hk_features[col]
    )

In [66]:
hk_features["hv_difference"] = (
    hk_features["cdtehvmon"] -
    hk_features["czthvmon"]
)

hk_features["hv_ratio"] = (
    hk_features["cdtehvmon"]
    /
    (hk_features["czthvmon"] + 1e-8)
)

In [67]:
hk_features["sun_angle"] = np.sqrt(

    hk_features["sun2yawdeg"]**2 +

    hk_features["sun2rolldeg"]**2 +

    hk_features["sun2pitchdeg"]**2

)

In [68]:
for col in [

    "sun2yawdeg",

    "sun2rolldeg",

    "sun2pitchdeg"

]:

    hk_features[f"{col}_gradient"] = np.gradient(

        hk_features[col]

    )

In [69]:
hk_features["czt_balance"] = (

    hk_features["czt1ctr"]

    -

    hk_features["czt2ctr"]

)

hk_features["cdte_balance"] = (

    hk_features["cdte1ctr"]

    -

    hk_features["cdte2ctr"]

)

In [70]:
hk_features["total_detector_counts"] = (

    hk_features["czt1ctr"]

    +

    hk_features["czt2ctr"]

    +

    hk_features["cdte1ctr"]

    +

    hk_features["cdte2ctr"]

)

In [71]:
for col in [

    "czt1ctr",

    "czt2ctr",

    "cdte1ctr",

    "cdte2ctr"

]:

    hk_features[f"{col}_rolling_mean"] = (

        hk_features[col]

        .rolling(WINDOW, min_periods=1)

        .mean()

    )

In [72]:
hk_features = hk_features.bfill().ffill()

print(

    hk_features.isna().sum().sum()

)

0


In [73]:
print(

    hk_features.shape

)

hk_features.describe().T

(5610, 87)


,count,mean,std,min,25%,50%,75%,max
l0recnum,5610.0,2805.500000,1619.611836,1.0,1403.25,2805.5,4207.75,5610.0
l0grtyr,5610.0,2026.000000,0.000000,2026.0,2026.00,2026.0,2026.00,2026.0
l0grtmon,5610.0,6.000000,0.000000,6.0,6.00,6.0,6.00,6.0
l0grtdy,5610.0,25.000000,0.000000,25.0,25.00,25.0,25.00,25.0
l0grthr,5610.0,6.000000,0.000000,6.0,6.00,6.0,6.00,6.0
...,...,...,...,...,...,...,...,...
total_detector_counts,5610.0,88.093048,36.717721,24.0,64.00,80.0,100.00,514.0
czt1ctr_rolling_mean,5610.0,44.925312,5.377644,31.8,41.00,44.3,48.40,65.1
czt2ctr_rolling_mean,5610.0,36.634879,5.218532,24.4,32.90,35.7,39.50,64.7
cdte1ctr_rolling_mean,5610.0,3.374326,3.482664,0.0,2.00,2.6,3.40,35.2


In [75]:
from scipy.signal import find_peaks

peak_indices, properties = find_peaks(
    features["COUNTS"],
    prominence=20,
    distance=20
)

features["is_peak"] = 0
features.loc[peak_indices, "is_peak"] = 1

print("Detected Peaks :", len(peak_indices))

Detected Peaks : 1522


In [76]:
features["peak_prominence"] = 0.0

features.loc[
    peak_indices,
    "peak_prominence"
] = properties["prominences"]

In [77]:
features["peak_height"] = 0.0

features.loc[
    peak_indices,
    "peak_height"
] = features.loc[
    peak_indices,
    "COUNTS"
]

In [78]:
from scipy.signal import peak_widths

widths = peak_widths(
    features["COUNTS"],
    peak_indices,
    rel_height=0.5
)

features["peak_width"] = 0.0

features.loc[
    peak_indices,
    "peak_width"
] = widths[0]

In [79]:
features["time_since_last_peak"] = np.nan

previous = None

for idx in peak_indices:

    if previous is None:
        features.loc[idx, "time_since_last_peak"] = 0

    else:
        delta = (
            features.loc[idx, "DATETIME"]
            -
            features.loc[previous, "DATETIME"]
        ).total_seconds()

        features.loc[idx, "time_since_last_peak"] = delta

    previous = idx

features["time_since_last_peak"] = (
    features["time_since_last_peak"]
    .ffill()
    .fillna(0)
)

In [80]:
features["rise_rate"] = (

    features["COUNTS"]

    -

    features["lag_1"]

)

In [81]:
features["decay_rate"] = (

    features["lag_1"]

    -

    features["COUNTS"]

)

In [82]:
WINDOW = 300

features["peak_density"] = (

    features["is_peak"]

    .rolling(WINDOW, min_periods=1)

    .sum()

)

In [83]:
features["integrated_counts"] = (

    features["COUNTS"]

    .rolling(30, min_periods=1)

    .sum()

)

In [84]:
features["peak_rank"] = (

    features["COUNTS"]

    /

    (
        features["rolling_max"]

        + 1e-8

    )

)

In [85]:
threshold = features["COUNTS"].quantile(0.99)

features["strong_peak"] = (

    features["COUNTS"] >= threshold

).astype(int)

In [86]:
flare_threshold = (

    features["rolling_mean"]

    +

    3 *

    features["rolling_std"]

)

features["flare_candidate"] = (

    features["COUNTS"]

    >

    flare_threshold

).astype(int)

In [87]:
features = features.bfill().ffill()

print(features.isna().sum().sum())

0


In [88]:
peak_cols = [

    "is_peak",

    "peak_prominence",

    "peak_height",

    "peak_width",

    "time_since_last_peak",

    "rise_rate",

    "decay_rate",

    "peak_density",

    "integrated_counts",

    "peak_rank",

    "strong_peak",

    "flare_candidate"

]

features[peak_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
is_peak,43188.0,0.035241,0.184391,0.0,0.0,0.0,0.0,1.0
peak_prominence,43188.0,4.334398,23.783093,0.0,0.0,0.0,0.0,362.0
peak_height,43188.0,4.334398,23.783093,0.0,0.0,0.0,0.0,362.0
peak_width,43188.0,0.036639,0.193614,0.0,0.0,0.0,0.0,2.0
time_since_last_peak,43188.0,28.361211,6.294188,0.0,23.0,25.0,32.0,43.0
rise_rate,43188.0,-0.005835,50.582760,-362.0,0.0,0.0,0.0,362.0
decay_rate,43188.0,0.005835,50.582760,-362.0,0.0,0.0,0.0,362.0
peak_density,43188.0,10.537557,1.032324,0.0,10.0,11.0,11.0,13.0
integrated_counts,43188.0,395.554876,92.330030,150.0,332.0,389.0,450.0,928.0
peak_rank,43188.0,0.102586,0.260050,0.0,0.0,0.0,0.0,1.0


In [90]:
print("="*50)
print(f"Lightcurve Features    : {features.shape[1]}")
print(f"Spectral Features      : {spectral_features.shape[1]}")
print(f"Housekeeping Features  : {hk_features.shape[1]}")
print("="*50)

total = (
    features.shape[1]
    + spectral_features.shape[1]
    + hk_features.shape[1]
)

print(f"Total Engineered Columns Across All DataFrames : {total}")

Lightcurve Features    : 42
Spectral Features      : 22
Housekeeping Features  : 87
Total Engineered Columns Across All DataFrames : 151


In [94]:
print(features.columns.tolist())
print(spectral_features.columns.tolist())
print(hk_features.columns.tolist())

['MJD', 'ISOT', 'COUNTS', 'STAT_ERR', 'DATETIME', 'rolling_mean', 'rolling_std', 'rolling_max', 'rolling_min', 'ema_10', 'ema_30', 'diff1', 'diff2', 'gradient', 'lag_1', 'lag_2', 'lag_5', 'lag_10', 'window_energy', 'window_variance', 'rolling_skew', 'rolling_kurtosis', 'rolling_rms', 'rolling_median', 'rolling_mad', 'rolling_cv', 'z_score', 'robust_z', 'peak_to_peak', 'local_energy', 'is_peak', 'peak_prominence', 'peak_height', 'peak_width', 'time_since_last_peak', 'rise_rate', 'decay_rate', 'peak_density', 'integrated_counts', 'peak_rank', 'strong_peak', 'flare_candidate']
['SPEC_NUM', 'ROWID', 'TSTART', 'TSTOP', 'EXPOSURE', 'MID_TIME', 'spec_total_counts', 'spec_mean_counts', 'spec_std_counts', 'spec_max', 'spec_min', 'peak_channel', 'spectral_centroid', 'spectral_spread', 'spectral_entropy', 'low_energy', 'mid_energy', 'high_energy', 'hardness_ratio1', 'hardness_ratio2', 'dominant_fraction', 'active_channels']
['l0recnum', 'l0grtyr', 'l0grtmon', 'l0grtdy', 'l0grthr', 'l0grtmin', 'l0